# NCEI CUDEMs in Cascadia
## (Washington Coast so far)

Under development for the [Cascadia CoPes Hub](https://cascadiacopeshub.org/) project, supported by NSF.

This is an attempt to catalog the CUDEM tiles available on NCEI servers, either as GeoTiff (.tif) files or as netCDF (.nc) files, showing the year released and whether they are referenced to NAVD88 or MHW.

:::{note}
The list of DEM sources below may not be complete. Additions/Corrections welcome.

It is assumed that a file is MHW only if `nthmp` is in the url. Is this always true?
:::

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools
from clawpack.visclaw import geoplot
from importlib import reload
import geopandas as gpd
import folium
import pandas as pd
from IPython.display import FileLink

In [ ]:
import find_topo_source

## Select a set of tiles

Specify the NW corner of the 0.25 x 0.25 degree tiles to search for:

In [ ]:
eighth = 1/8  # half tile width, 0.0125 degrees

yNWs = arange(46.5, 48.6, 0.25)
xNWs = arange(-125.0, -123.4, 0.25)
    
# make list of tile names (north to south):
tile_names = []
for yNW in yNWs[-1::-1]:
    for xNW in xNWs:
        # midpoint of tile is (xm,ym):
        xm = xNW+eighth
        ym = yNW-eighth
        tile_name = find_topo_source.tile_coords(xm,ym,verbose=False)
        #print(f'midpoints: {xm:.3f}, {ym:.3f}, NW corner:  {x:.3f}, {y:.3f}, tile: {tile_name}')
        tile_names.append(tile_name)

## Search for tiles in catlogs:

In [ ]:
%%capture tile_sources_summary

print('Versions of tiles found in these catalogs:')
versions_mhw = {}
versions_navd88 = {}
for tile_name in tile_names:
    versions_mhw[tile_name] = []
    versions_navd88[tile_name] = []
    tile_urls = find_topo_source.find_tile_url(tile_name, verbose=False)
    for url in tile_urls:
        if url[-3:]=='.nc':
            vinfo = url[-9:]
        else:
            vinfo = url[-10:]
        if 'v' not in vinfo:
            vinfo = vinfo[2:]
        if 'nthmp' in url:
            versions_mhw[tile_name].append(vinfo)
        else:
            versions_navd88[tile_name].append(vinfo)
    print(f'    {tile_name}, MHW: {str(versions_mhw[tile_name]):>14}' \
            +f'   NAVD88: {str(versions_navd88[tile_name])}')

**The summary generated is printed out below the map.**

## Map of tiles

In [ ]:
m = folium.Map(location=(47.5,-125), zoom_start=8, height=800, scrollWheelZoom=False)

for y in yNWs[-1::-1]:
    for x in xNWs:
        xm = x+eighth
        ym = y-eighth
        tile_name = find_topo_source.tile_coords(xm,ym,verbose=False)
        popup = f'<b>{tile_name}</b>'
        for vinfo in versions_mhw[tile_name]:
            popup = popup + f'\n<font color="green">{vinfo}</font>'
        for vinfo in versions_navd88[tile_name]:
            popup = popup + f'\n<font color="blue">{vinfo}</font>'
        folium.Marker(
            location=[y-eighth,x+eighth],
            popup = popup,
            tooltip = "Click for info",
            #tooltip = f"<b>Tile:</b>\n {tile_name}",
            icon=folium.Icon(color="red", icon="info-sign") # Customize the marker's appearance
        ).add_to(m) 
        x1 = x
        x2 = x + 0.25
        y1 = y - 0.25
        y2 = y

        found_nc = array(['.nc' in v for v in versions[tile_name]]).sum() > 0
        found_tif = array(['.tif' in v for v in versions[tile_name]]).sum() > 0
        if len(versions_mhw[tile_name])>0 and len(versions_navd88[tile_name])>0:
            color = 'green'
            tip = 'MHW and NAVD88'
        elif len(versions_navd88[tile_name])>0:
            color = 'blue'
            tip = 'NAVD88 only'
        else:
            color = 'red'
            tip = 'not found'
        
        folium.Polygon(
            locations=[[y1,x1], [y1,x2], [y2,x2], [y2,x1]],
            color="black", # Outline color
            weight=1,
            fill=True,
            fillColor=color,
            fillOpacity=0.2,
            tooltip=tip
        ).add_to(m)

for lat in arange(yNWs[0]-0.25, yNWs[-1]+.1, 0.25):
    folium.PolyLine(
           [[lat,-125.25], [lat,-123.25]],
           color='black', weight=1, opacity=0.5
       ).add_to(m)

    #Add label at edge
    folium.Marker(
        [lat+0.05, -125.25],
        icon=folium.DivIcon(html=f'<div style="font-size: 12pt; color: gray;">{lat}°</div>')
        ).add_to(m)

# save as stand-alone html file:
fname = 'WA-CUDEM-Tiles-Map.html'
m.save(fname)
print('Created ',fname)

# display map inline:
m

The stand-alone map can be viewed at [interactive map](SoWA-Tiles-Map.html).

Clicking on an info marker shows a summary of what files are available (in green if MHW, blue if NAVD88).

### Tabular summary of what file versions were found:

In [ ]:
tile_sources_summary()

#### Print this table to a file...

In [ ]:
tile_versions_summary_fname = 'WAcoast-CUDEM-Tiles-Summary.txt'
with open(tile_versions_summary_fname,'w') as tv_file:
    tv_file.write(tile_sources_summary.stdout)
print('Created ', tile_versions_summary_fname)

See [WAcoast-CUDEM-Tiles-Summary.txt](WAcoast-CUDEM-Tiles-Summary.txt) for the output.

## Versions of these tiles found on NCEI webpages

Print out more details of which catalogs the above tiles are found in, and their URLs:

(Note that some of the .nc files are labelled as NAVD88 if their URL does not contain `nthmp`. Is this always correct?)

In [ ]:
%%capture tile_sources_text

print('CUDEM tiles found on the WA Coast:\n')
tile_names = []
for y in yNWs[-1::-1]:
    print('\n=====================')
    print(f'Latitude y = {y:.2f}:')
    print('=====================')

    for x in xNWs:
        xm = x+eighth
        ym = y-eighth
        tile_name = find_topo_source.tile_coords(xm,ym,verbose=False)
        print('\n------------------------------------------------')
        print(f'Tile: {tile_name},  NW corner:  {x:.3f}, {y:.3f}')
        tile_names.append(tile_name)
        tile_urls = find_topo_source.find_tile_url(tile_name, verbose=True)
        if len(tile_urls) == 0:
            print(f'   Not found')


In [ ]:
tile_versions_fname = 'WAcoast-CUDEM-Tiles.txt'
with open(tile_versions_fname,'w') as tv_file:
    tv_file.write(tile_sources_text.stdout)
print('Created ', tile_versions_fname)

See [WAcoast-CUDEM-Tiles.txt](WAcoast-CUDEM-Tiles.txt) for the output, or execute the next cell to see the output here...

In [ ]:
#tile_sources_text()